In [46]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd()
if not (data_dir / "market_returns_2000_2025.csv").is_file():
    data_dir = data_dir / "CAPM Factors"

market = pd.read_csv(data_dir / "market_returns_2000_2025.csv", parse_dates=["date"])
risk_free = pd.read_csv(data_dir / "risk_free_rate_2000_2025.csv", parse_dates=["date"])

monthly = market.merge(
    risk_free,
    on="date",
    how="outer",
    validate="one_to_one",
    indicator=True,
)
unmatched = monthly.loc[monthly["_merge"] != "both", "date"]
if not unmatched.empty:
    raise ValueError(f"Market and risk-free CSV dates do not match: {unmatched.tolist()[:5]}")
if monthly[["market_return", "rf_1m"]].isna().any().any():
    raise ValueError("Market return or risk-free rate contains missing values.")

factors = pd.DataFrame(
    {
        "Mkt-RF": (monthly["market_return"] - monthly["rf_1m"]).to_numpy(),
        "RF": monthly["rf_1m"].to_numpy(),
    },
    index=pd.DatetimeIndex(monthly["date"], name="Date"),
).sort_index()



## Clean S&P 500 monthly stock returns

Use the supplied `total_return` field as the monthly total return (do not recompute it from price). Remove missing returns and exact duplicate `PERMNO`/date records, and winsorize the cross-section of returns within each month at the 1st and 99th percentiles. Keep cleaned long and wide panels in memory only. `PERMNO` identifies securities because ticker symbols can be reused.

In [47]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd()
if not (data_dir / "sp500_stocks_2000_2025.csv").is_file():
    data_dir = data_dir / "CAPM Factors"

source_path = data_dir / "sp500_stocks_2000_2025.csv"
required_columns = {"date", "permno", "ticker", "company_name", "total_return"}
source_raw = pd.read_csv(source_path, parse_dates=["date"])
stocks = source_raw.copy()
missing_columns = required_columns.difference(stocks.columns)
if missing_columns:
    raise ValueError(f"Stock data is missing required columns: {sorted(missing_columns)}")

source_rows = len(stocks)
stocks["date"] = pd.to_datetime(stocks["date"], errors="coerce")
if stocks["date"].isna().any() or stocks["permno"].isna().any():
    raise ValueError("Stock data contains missing or invalid dates or PERMNO identifiers.")
if not stocks["date"].dt.is_month_end.all():
    raise ValueError("Expected month-end observations in the stock return file.")
stocks["total_return"] = pd.to_numeric(stocks["total_return"], errors="raise")

key_columns = ["permno", "date"]
duplicate_rows = stocks.duplicated(key_columns, keep=False)
if duplicate_rows.any():
    duplicate_groups = stocks.loc[duplicate_rows].groupby(key_columns, dropna=False)
    conflicting_duplicates = duplicate_groups.nunique(dropna=False).gt(1).any(axis=1)
    if conflicting_duplicates.any():
        raise ValueError("Conflicting duplicate PERMNO/date observations found.")
duplicates_removed = int(stocks.duplicated(key_columns).sum())
stocks = stocks.drop_duplicates(key_columns).copy()

missing_returns = int(stocks["total_return"].isna().sum())
stocks = stocks.dropna(subset=["total_return"]).copy()

stocks["return_winsorized"] = stocks.groupby("date")["total_return"].transform(
    lambda monthly_returns: monthly_returns.clip(
        lower=monthly_returns.quantile(0.01),
        upper=monthly_returns.quantile(0.99),
    )
)
winsorized_returns = int(stocks["return_winsorized"].ne(stocks["total_return"]).sum())

monthly_returns = stocks[
    ["date", "permno", "ticker", "company_name", "total_return", "return_winsorized"]
].sort_values(["date", "permno"])
wide_returns = monthly_returns.pivot(
    index="date", columns="permno", values="return_winsorized"
).sort_index()
wide_returns.index.name = "date"
wide_returns.columns = [f"permno_{int(permno)}" for permno in wide_returns.columns]



## 60-month rolling beta estimates

For each stock and month-end, estimate ordinary beta using all paired stock and market excess returns in the trailing 60 calendar months, and downside beta using only paired observations where market excess return is strictly negative. Both estimates use the supplied monthly `total_return` and the aligned market and risk-free CSVs in decimal units; they do not use the winsorized return column. Missing observations are excluded, windows require at least 12 valid observations, and a missing/zero market variance yields NaN. Each estimate dated t is available after month t closes and is used to predict the next return month. The stock file provides total returns but no separate delisting-return field, so no additional delisting adjustment is applied.

In [48]:
import numpy as np
import pandas as pd

WINDOW_MONTHS = 60
MIN_OBSERVATIONS = 12

calendar = pd.DatetimeIndex(factors.index, name="date")
expected_calendar = pd.date_range(
    start=calendar.min(), end=calendar.max(), freq=pd.offsets.MonthEnd()
)
if not calendar.equals(expected_calendar):
    raise ValueError("Market data must contain one continuous observation per month-end.")
market_excess = pd.Series(factors["Mkt-RF"].to_numpy(), index=calendar)
rf_by_month = pd.Series(factors["RF"].to_numpy(), index=calendar)
full_window = pd.Series(1, index=calendar).rolling(
    WINDOW_MONTHS, min_periods=1
).count().eq(WINDOW_MONTHS)

stock_data = monthly_returns.copy()
if stock_data.duplicated(["permno", "date"]).any():
    raise ValueError("Cleaned stock data contains duplicate PERMNO/month observations.")
if not stock_data["date"].isin(calendar).all():
    raise ValueError("Stock-return dates do not all align with market months.")

results = []
for permno, stock in stock_data.groupby("permno", sort=False):
    stock_returns = stock.set_index("date")["total_return"].reindex(calendar)
    stock_excess = stock_returns - rf_by_month
    paired = stock_excess.notna() & market_excess.notna()
    downside = paired & market_excess.lt(0)

    def rolling_slope(mask):
        count = mask.astype("int64").rolling(
            WINDOW_MONTHS, min_periods=1
        ).sum()
        x = stock_excess.where(mask, 0.0)
        y = market_excess.where(mask, 0.0)
        sum_x = x.rolling(WINDOW_MONTHS, min_periods=1).sum()
        sum_y = y.rolling(WINDOW_MONTHS, min_periods=1).sum()
        sum_xy = (x * y).rolling(WINDOW_MONTHS, min_periods=1).sum()
        sum_y_squared = (y.pow(2)).rolling(
            WINDOW_MONTHS, min_periods=1
        ).sum()
        centered_cross_product = sum_xy - sum_x * sum_y / count
        centered_market_sum_squares = sum_y_squared - sum_y.pow(2) / count
        beta = centered_cross_product / centered_market_sum_squares
        beta = beta.where(
            full_window & (count >= MIN_OBSERVATIONS) & (centered_market_sum_squares > 0)
        )
        return beta, count

    beta, valid_count = rolling_slope(paired)
    downside_beta, downside_count = rolling_slope(downside)

    # Emit an estimate for each stock-month with an observed monthly stock return.
    estimation_months = pd.DatetimeIndex(stock["date"].sort_values().unique())
    theoretical_start = estimation_months - pd.offsets.MonthEnd(WINDOW_MONTHS - 1)
    window_start = pd.DatetimeIndex([max(start, calendar[0]) for start in theoretical_start])
    results.append(
        pd.DataFrame(
            {
                "permno": int(permno),
                "estimation_month": estimation_months,
                "beta": beta.reindex(estimation_months).to_numpy(),
                "downside_beta": downside_beta.reindex(estimation_months).to_numpy(),
                "n_observations": valid_count.reindex(estimation_months).to_numpy(dtype="int64"),
                "n_downside_observations": downside_count.reindex(estimation_months).to_numpy(dtype="int64"),
                "window_start": window_start,
                "window_end": estimation_months,
            }
        )
    )

rolling_betas = pd.concat(results, ignore_index=True).sort_values(
    ["estimation_month", "permno"]
)



## Company-level downside-beta and market-factor regressions: 2009-2020 and COVID-19

For each security (PERMNO), estimate conventional time-series OLS in two return windows: July 2009-January 2020 and the COVID-19 shock, February-December 2020. The dependent variable is same-month stock excess return: supplied stock total return minus same-month risk-free rate.

- **Model 1:** stock excess return = intercept + coefficient on rolling downside beta.
- **Model 2:** stock excess return = intercept + coefficient on market excess return (`Mkt-RF`) + coefficient on rolling downside beta.

Model 1 uses rolling downside beta only; Model 2 adds market excess return as a control. Predictors are matched to the same month as returns, without an additional lag. Use conventional OLS standard errors, Student-t p-values, and a minimum of 10 valid observations per security-model. The next cell repeats both models on pooled stock-month observations rather than fitting one model per PERMNO.

Timing limitation: rolling downside beta uses a window that includes the same month's stock return. These are contemporaneous associations, not clean predictive or causal tests. PERMNO is the security unit; individual-company significance counts are unadjusted for multiple testing.

In [54]:
import math
import numpy as np
import pandas as pd
from IPython.display import display


def _beta_cf(a, b, x):
    max_iterations, epsilon, tiny = 200, 3e-14, 1e-300
    qab, qap, qam = a + b, a + 1, a - 1
    c = 1.0
    d = 1.0 - qab * x / qap
    if abs(d) < tiny:
        d = tiny
    d = 1.0 / d
    h = d
    for iteration in range(1, max_iterations + 1):
        m2 = 2 * iteration
        aa = iteration * (b - iteration) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < tiny:
            d = tiny
        c = 1.0 + aa / c
        if abs(c) < tiny:
            c = tiny
        d = 1.0 / d
        h *= d * c
        aa = -(a + iteration) * (qab + iteration) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < tiny:
            d = tiny
        c = 1.0 + aa / c
        if abs(c) < tiny:
            c = tiny
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < epsilon:
            return h
    raise ArithmeticError("Incomplete-beta evaluation did not converge.")


def _regularized_beta(x, a, b):
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    factor = math.exp(
        math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
        + a * math.log(x) + b * math.log1p(-x)
    )
    if x < (a + 1) / (a + b + 2):
        return factor * _beta_cf(a, b, x) / a
    return 1 - factor * _beta_cf(b, a, 1 - x) / b


def _student_t_pvalue(t_stat, df):
    return _regularized_beta(df / (df + t_stat * t_stat), df / 2, 0.5)


def _student_t_critical(alpha, df):
    low, high = 0.0, 100.0
    for _ in range(80):
        middle = (low + high) / 2
        if _student_t_pvalue(middle, df) > alpha:
            low = middle
        else:
            high = middle
    return (low + high) / 2


MIN_REGRESSION_OBSERVATIONS = 10
SIGNIFICANCE_LEVEL = 0.05
PERIODS = {
    "Between crises (July 2009-January 2020)": (pd.Timestamp("2009-07-31"), pd.Timestamp("2020-01-31")),
    "COVID-19 shock (February-December 2020)": (pd.Timestamp("2020-02-29"), pd.Timestamp("2020-12-31")),
}
MODEL_SPECS = {
    "Model 1: rolling downside beta only": ["downside_beta"],
    "Model 2: market excess return + rolling downside beta": ["Mkt-RF", "downside_beta"],
}

# Align returns, RF, market excess returns, and downside beta by the same month.
returns = monthly_returns[["permno", "ticker", "company_name", "date", "total_return"]].copy()
returns = returns.rename(columns={"date": "return_month", "total_return": "stock_return"})
predictive = returns.merge(
    rolling_betas[["permno", "estimation_month", "downside_beta"]],
    left_on=["permno", "return_month"],
    right_on=["permno", "estimation_month"],
    how="left",
    validate="one_to_one",
)
market_factors = factors[["Mkt-RF", "RF"]].rename_axis("return_month").reset_index()
predictive = predictive.merge(
    market_factors, on="return_month", how="left", validate="many_to_one"
)
if predictive[["Mkt-RF", "RF"]].isna().any().any():
    raise ValueError("A return month is missing its market excess return or risk-free rate.")
predictive["excess_return"] = predictive["stock_return"] - predictive["RF"]
predictive = predictive.replace([np.inf, -np.inf], np.nan)

coefficient_rows = []
model_rows = []
for period_name, (period_start, period_end) in PERIODS.items():
    period_data = predictive.loc[
        predictive["return_month"].between(period_start, period_end)
    ]
    if period_data.empty:
        raise ValueError(f"No observations found for {period_name}.")

    for permno, stock in period_data.groupby("permno", sort=False):
        metadata = stock.iloc[0]
        for model_name, regressors in MODEL_SPECS.items():
            sample = stock.dropna(subset=["excess_return", *regressors])
            n = len(sample)
            k = len(regressors) + 1
            df = n - k
            if n < MIN_REGRESSION_OBSERVATIONS or df <= 0:
                continue
            design = np.column_stack([
                np.ones(n), sample[regressors].to_numpy(dtype=float)
            ])
            if np.linalg.matrix_rank(design) < k:
                continue
            y = sample["excess_return"].to_numpy(dtype=float)
            coefficients, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
            residuals = y - design @ coefficients
            sse = float(residuals @ residuals)
            residual_variance = sse / df
            covariance = residual_variance * np.linalg.inv(design.T @ design)
            standard_errors = np.sqrt(np.maximum(np.diag(covariance), 0))
            critical = _student_t_critical(SIGNIFICANCE_LEVEL, df)

            for j, term in enumerate(["alpha", *regressors]):
                estimate = coefficients[j]
                se = standard_errors[j]
                t_stat = estimate / se if se > 0 else np.nan
                p_value = _student_t_pvalue(abs(t_stat), df) if np.isfinite(t_stat) else np.nan
                coefficient_rows.append({
                    "period": period_name,
                    "permno": permno,
                    "ticker": metadata["ticker"],
                    "company_name": metadata["company_name"],
                    "model": model_name,
                    "term": term,
                    "coefficient": estimate,
                    "ols_std_error": se,
                    "t_stat": t_stat,
                    "degrees_of_freedom": df,
                    "p_value": p_value,
                    "ci_95_low": estimate - critical * se,
                    "ci_95_high": estimate + critical * se,
                    "n_months": n,
                    "significant_5pct": bool(np.isfinite(p_value) and p_value < SIGNIFICANCE_LEVEL),
                })

            centered_y = y - y.mean()
            sst = float(centered_y @ centered_y)
            if sst > 0:
                r_squared = 1 - sse / sst
                if sse <= 0:
                    f_stat, f_p = float("inf"), 0.0
                else:
                    f_stat = (r_squared / len(regressors)) / ((1 - r_squared) / df)
                    f_x = df / (df + len(regressors) * f_stat)
                    f_p = _regularized_beta(f_x, df / 2, len(regressors) / 2)
                model_rows.append({
                    "period": period_name,
                    "permno": permno,
                    "model": model_name,
                    "n_months": n,
                    "r_squared": r_squared,
                    "f_stat": f_stat,
                    "model_p_value": f_p,
                    "model_significant_5pct": bool(f_p < SIGNIFICANCE_LEVEL),
                })

firm_regression_results = pd.DataFrame(coefficient_rows)
firm_model_results = pd.DataFrame(model_rows)
if firm_regression_results.empty or firm_model_results.empty:
    raise ValueError("No firm-level regressions met the sample and rank requirements.")

slope_results = firm_regression_results.loc[firm_regression_results["term"].ne("alpha")]
summary_rows = []
for (period, model, term), group in slope_results.groupby(["period", "model", "term"]):
    significant = group.loc[group["significant_5pct"]]
    summary_rows.append({
        "period": period,
        "model": model,
        "coefficient": term,
        "eligible_securities": group["permno"].nunique(),
        "significant_securities_5pct": significant["permno"].nunique(),
        "significant_positive": significant.loc[significant["coefficient"].gt(0), "permno"].nunique(),
        "significant_negative": significant.loc[significant["coefficient"].lt(0), "permno"].nunique(),
        "significant_percent": 100 * significant["permno"].nunique() / group["permno"].nunique(),
        "mean_coefficient": group["coefficient"].mean(),
        "median_coefficient": group["coefficient"].median(),
    })
coefficient_summary = pd.DataFrame(summary_rows)
model_significance_summary = (
    firm_model_results.groupby(["period", "model"], as_index=False)
    .agg(
        eligible_securities=("permno", "nunique"),
        significant_models_5pct=("model_significant_5pct", "sum"),
        mean_r_squared=("r_squared", "mean"),
        median_r_squared=("r_squared", "median"),
    )
)
model_significance_summary["significant_percent"] = (
    100 * model_significance_summary["significant_models_5pct"]
    / model_significance_summary["eligible_securities"]
)

print("Model 1 regressor: rolling downside beta only.")
print("Model 2 regressors: market excess return (Mkt-RF) and rolling downside beta.")
print("Beta and return observations are matched in the same month; conventional OLS inference.")
print("Individual coefficient significance:")
display(coefficient_summary)
print("Joint F-tests against intercept-only model:")
display(model_significance_summary)
print("Full security-level coefficient results are in firm_regression_results.")

Model 1 regressor: rolling downside beta only.
Model 2 regressors: market excess return (Mkt-RF) and rolling downside beta.
Beta and return observations are matched in the same month; conventional OLS inference.
Individual coefficient significance:
                                    period  ... median_coefficient
0  Between crises (July 2009-January 2020)  ...           0.000145
1  Between crises (July 2009-January 2020)  ...           1.065698
2  Between crises (July 2009-January 2020)  ...           0.000792
3  COVID-19 shock (February-December 2020)  ...           0.166441
4  COVID-19 shock (February-December 2020)  ...           1.005044
5  COVID-19 shock (February-December 2020)  ...          -0.022995

[6 rows x 10 columns]
Joint F-tests against intercept-only model:
                                    period  ... significant_percent
0  Between crises (July 2009-January 2020)  ...            2.772964
1  Between crises (July 2009-January 2020)  ...           89.774697
2  COVID-19

period,model,coefficient,eligible_securities,significant_securities_5pct,significant_positive,significant_negative,significant_percent,mean_coefficient,median_coefficient
Between crises (July 2009-January 2020),Model 1: rolling downside beta only,downside_beta,577,16,10,6,2.772964,0.048689,0.000145
Between crises (July 2009-January 2020),Model 2: market excess return + rolling downside beta,Mkt-RF,577,527,527,0,91.334489,1.083003,1.065698
Between crises (July 2009-January 2020),Model 2: market excess return + rolling downside beta,downside_beta,577,22,8,14,3.812825,0.037465,0.000792
COVID-19 shock (February-December 2020),Model 1: rolling downside beta only,downside_beta,398,14,7,7,3.517588,0.163956,0.166441
COVID-19 shock (February-December 2020),Model 2: market excess return + rolling downside beta,Mkt-RF,398,298,298,0,74.874372,1.122488,1.005044
COVID-19 shock (February-December 2020),Model 2: market excess return + rolling downside beta,downside_beta,398,10,4,6,2.512563,0.014533,-0.022995


period,model,eligible_securities,significant_models_5pct,mean_r_squared,median_r_squared,significant_percent
Between crises (July 2009-January 2020),Model 1: rolling downside beta only,577,16,0.013297,0.005071,2.772964
Between crises (July 2009-January 2020),Model 2: market excess return + rolling downside beta,577,518,0.311832,0.313580,89.774697
COVID-19 shock (February-December 2020),Model 1: rolling downside beta only,398,14,0.117889,0.089476,3.517588
COVID-19 shock (February-December 2020),Model 2: market excess return + rolling downside beta,398,287,0.627452,0.674056,72.110553


## Pooled OLS across all stock-month observations

This pooled analysis stacks the eligible stock-month rows from each selected period into one large OLS regression per model, using the same formulas and same-month matching as the company-level analysis. Model 1 regresses stock excess return on an intercept and rolling downside beta; Model 2 uses an intercept, market excess return (`Mkt-RF`), and rolling downside beta.

The reported coefficient p-values use conventional OLS standard errors and are included to mirror the requested ordinary OLS comparison. Because all stocks in a month share the same market return and stock residuals can be correlated across firms, pooled conventional standard errors assume independence that is unlikely to hold; treat pooled significance as descriptive and potentially overstated. This pooled regression is different from the per-company time-series regressions above.

In [55]:
import numpy as np
import pandas as pd
from IPython.display import display

pooled_coefficient_rows = []
pooled_model_rows = []
for period_name, (period_start, period_end) in PERIODS.items():
    period_data = predictive.loc[
        predictive["return_month"].between(period_start, period_end)
    ].copy()

    for model_name, regressors in MODEL_SPECS.items():
        sample = period_data.dropna(subset=["excess_return", *regressors]).copy()
        n_observations = len(sample)
        n_parameters = len(regressors) + 1
        degrees_of_freedom = n_observations - n_parameters
        if degrees_of_freedom <= 0:
            raise ValueError(f"Too few pooled observations for {period_name}, {model_name}.")

        design = np.column_stack([
            np.ones(n_observations),
            sample[regressors].to_numpy(dtype=float),
        ])
        if np.linalg.matrix_rank(design) < n_parameters:
            raise ValueError(f"Pooled design matrix is rank deficient for {period_name}, {model_name}.")

        y = sample["excess_return"].to_numpy(dtype=float)
        coefficients, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
        residuals = y - design @ coefficients
        residual_sum_squares = float(residuals @ residuals)
        residual_variance = residual_sum_squares / degrees_of_freedom
        covariance = residual_variance * np.linalg.inv(design.T @ design)
        standard_errors = np.sqrt(np.maximum(np.diag(covariance), 0))
        critical_value = _student_t_critical(SIGNIFICANCE_LEVEL, degrees_of_freedom)

        for position, term in enumerate(["alpha", *regressors]):
            estimate = coefficients[position]
            standard_error = standard_errors[position]
            t_stat = estimate / standard_error if standard_error > 0 else np.nan
            p_value = (
                _student_t_pvalue(abs(t_stat), degrees_of_freedom)
                if np.isfinite(t_stat)
                else np.nan
            )
            pooled_coefficient_rows.append({
                "period": period_name,
                "model": model_name,
                "term": term,
                "coefficient": estimate,
                "ols_std_error": standard_error,
                "t_stat": t_stat,
                "degrees_of_freedom": degrees_of_freedom,
                "p_value": p_value,
                "ci_95_low": estimate - critical_value * standard_error,
                "ci_95_high": estimate + critical_value * standard_error,
                "significant_5pct": bool(np.isfinite(p_value) and p_value < SIGNIFICANCE_LEVEL),
                "n_stock_months": n_observations,
                "n_securities": sample["permno"].nunique(),
            })

        centered_y = y - y.mean()
        total_sum_squares = float(centered_y @ centered_y)
        r_squared = (
            1 - residual_sum_squares / total_sum_squares
            if total_sum_squares > 0
            else np.nan
        )
        if total_sum_squares > 0 and residual_sum_squares > 0:
            f_stat = (r_squared / len(regressors)) / (
                (1 - r_squared) / degrees_of_freedom
            )
            f_x = degrees_of_freedom / (
                degrees_of_freedom + len(regressors) * f_stat
            )
            f_p_value = _regularized_beta(
                f_x, degrees_of_freedom / 2, len(regressors) / 2
            )
        elif total_sum_squares > 0:
            f_stat, f_p_value = float("inf"), 0.0
        else:
            f_stat, f_p_value = np.nan, np.nan
        pooled_model_rows.append({
            "period": period_name,
            "model": model_name,
            "n_stock_months": n_observations,
            "n_securities": sample["permno"].nunique(),
            "r_squared": r_squared,
            "f_stat": f_stat,
            "joint_model_p_value": f_p_value,
            "joint_model_significant_5pct": bool(
                np.isfinite(f_p_value) and f_p_value < SIGNIFICANCE_LEVEL
            ),
        })

pooled_regression_results = pd.DataFrame(pooled_coefficient_rows)
pooled_model_summary = pd.DataFrame(pooled_model_rows)
print("Pooled OLS coefficient estimates, using all eligible stock-month observations:")
display(pooled_regression_results)
print("Pooled model joint F-tests against intercept-only model:")
display(pooled_model_summary)
print("Caution: conventional pooled OLS errors do not account for within-month cross-stock dependence.")

--- Executing cell 0 ---
--- Executing cell 2 ---
--- Executing cell 4 ---
--- Executing cell 6 ---
Model 1 regressor: rolling downside beta only.
Model 2 regressors: market excess return (Mkt-RF) and rolling downside beta.
Beta and return observations are matched in the same month; conventional OLS inference.
Individual coefficient significance:
                                    period  ... median_coefficient
0  Between crises (July 2009-January 2020)  ...           0.000145
1  Between crises (July 2009-January 2020)  ...           1.065698
2  Between crises (July 2009-January 2020)  ...           0.000792
3  COVID-19 shock (February-December 2020)  ...           0.166441
4  COVID-19 shock (February-December 2020)  ...           1.005044
5  COVID-19 shock (February-December 2020)  ...          -0.022995

[6 rows x 10 columns]
Joint F-tests against intercept-only model:
                                    period  ... significant_percent
0  Between crises (July 2009-January 2020)  ...

period,model,term,coefficient,ols_std_error,t_stat,degrees_of_freedom,p_value,ci_95_low,ci_95_high,significant_5pct,n_stock_months,n_securities
Between crises (July 2009-January 2020),Model 1: rolling downside beta only,alpha,0.012477,0.000595,20.970537,55790,2.897600e-97,0.011311,0.013643,True,55792,613
Between crises (July 2009-January 2020),Model 1: rolling downside beta only,downside_beta,0.000146,0.000449,0.325284,55790,7.449671e-01,-0.000735,0.001027,False,55792,613
Between crises (July 2009-January 2020),Model 2: market excess return + rolling downside beta,alpha,0.000090,0.000521,0.172586,55789,8.629771e-01,-0.000931,0.001110,False,55792,613
Between crises (July 2009-January 2020),Model 2: market excess return + rolling downside beta,Mkt-RF,1.062451,0.007655,138.786770,55789,0.000000e+00,1.047447,1.077455,True,55792,613
Between crises (July 2009-January 2020),Model 2: market excess return + rolling downside beta,downside_beta,-0.000167,0.000387,-0.429794,55789,6.673474e-01,-0.000926,0.000593,False,55792,613
COVID-19 shock (February-December 2020),Model 1: rolling downside beta only,alpha,0.007979,0.003966,2.011884,4511,4.429153e-02,0.000204,0.015754,True,4513,432
COVID-19 shock (February-December 2020),Model 1: rolling downside beta only,downside_beta,0.005532,0.002832,1.953418,4511,5.083183e-02,-0.000020,0.011083,False,4513,432
COVID-19 shock (February-December 2020),Model 2: market excess return + rolling downside beta,alpha,-0.012674,0.002962,-4.278991,4510,1.916348e-05,-0.018481,-0.006867,True,4513,432
COVID-19 shock (February-December 2020),Model 2: market excess return + rolling downside beta,Mkt-RF,1.140419,0.018784,60.712197,4510,0.000000e+00,1.103593,1.177245,True,4513,432
COVID-19 shock (February-December 2020),Model 2: market excess return + rolling downside beta,downside_beta,0.001405,0.002102,0.668356,4510,5.039403e-01,-0.002716,0.005526,False,4513,432


period,model,n_stock_months,n_securities,r_squared,f_stat,joint_model_p_value,joint_model_significant_5pct
Between crises (July 2009-January 2020),Model 1: rolling downside beta only,55792,613,0.000002,0.105810,0.744967,False
Between crises (July 2009-January 2020),Model 2: market excess return + rolling downside beta,55792,613,0.256651,9630.954998,0.000000,True
COVID-19 shock (February-December 2020),Model 1: rolling downside beta only,4513,432,0.000845,3.815842,0.050832,False
COVID-19 shock (February-December 2020),Model 2: market excess return + rolling downside beta,4513,432,0.450195,1846.451911,0.000000,True
